# Fundamentos de Ingeniería de Datos — Actividad 3
## ALMACENAMIENTO: prueba de carga y consulta

**Equipo:** Bethzy Sarahi Salcido Toro (220443324) · Guillermo Fernando Ornelas Jimenez (225361237)

Evidencia **E3.2 — Prueba de carga y consulta**. Guardamos los mismos datos de dos formas y medimos **carga**, **disco** y **consulta**:

| Forma | Estrategia |
|---|---|
| Archivo Parquet | Data lake (y lakehouse cuando DuckDB lo consulta directamente) |
| Tabla en DuckDB | Data warehouse |

Al final se mide una consulta con JOIN sobre el **esquema estrella** de `serving.db` (datos reales, sin multiplicar). La prueba se repite para los **3 datasets** con la misma función `medir(etiqueta)` y genera `Modelado_de_Datos/Reportes/mediciones_<etiqueta>.csv`.

### Configuración del entorno

In [1]:
# ── Importar tools.py (nuestro archivo de utilidades) ─────────────────────
import sys, os, gc, time
sys.path.append(os.path.expanduser('~/Documents/IA/4_semestre/ing_de_datos/ing_datos/tools'))
import tools

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)

A = tools.almacenamiento
os.makedirs(tools.RUTA_MODELADO_PRUEBAS, exist_ok=True)
os.makedirs(tools.RUTA_MODELADO_REPORTES, exist_ok=True)

print(f'DuckDB {duckdb.__version__} | pandas {pd.__version__}')
print(f'Serving : {tools.SERVING_DB.relative_to(tools.BASE_DIR)}')
print(f'Pruebas : {tools.RUTA_MODELADO_PRUEBAS.relative_to(tools.BASE_DIR)}')
print(f'Reportes: {tools.RUTA_MODELADO_REPORTES.relative_to(tools.BASE_DIR)}')

DuckDB 1.5.6 | pandas 2.3.3
Serving : Modelado_de_Datos/serving.db
Pruebas : Modelado_de_Datos/Pruebas
Reportes: Modelado_de_Datos/Reportes


---
## PASO 1 — Datos de prueba

Repetimos los datos **MULTIPLICAR** veces para que se noten las diferencias (con 50 mil filas todo tarda milésimas). Con 20 se obtienen entre 0.9 y 1.4 millones de filas por dataset, suficiente para medir sin agotar la memoria de la computadora. Usa 1 para tus datos reales.

In [2]:
MULTIPLICAR = 20                                                  # ← 1 = datos reales

def datos_prueba(et):
    """Lee el Parquet TRANSFORMED y repite sus filas MULTIPLICAR veces."""
    con_mem = duckdb.connect()                                    # DuckDB en memoria, solo para preparar
    df = con_mem.sql(f"""
        SELECT b.*
        FROM read_parquet('{tools.modelado.parquet(et)}') AS b, range({MULTIPLICAR})
    """).df()
    con_mem.close()
    return df

---
## PASOS 2 a 6 — Carga, disco, consulta y tabla de mediciones

La función `medir(etiqueta)` sigue los pasos de la práctica de clase, para un dataset:

| Paso | Qué mide | Cómo |
|---|---|---|
| 2 | ¿Cuánto tarda en guardarse? | `to_parquet()` contra `CREATE TABLE … AS SELECT` en DuckDB |
| 3 | ¿Cuánto ocupa? | Tamaño en MB de cada archivo (después de `CHECKPOINT`) |
| 4 | ¿Cuánto tarda en responder? | La **misma** consulta en los dos; los totales deben coincidir |
| 5 | Consulta sobre el esquema estrella | JOIN `fact_salarios_<etiqueta>` + `dim_experiencia_<etiqueta>` en `serving.db` (solo lectura) |
| 6 | Tabla de mediciones | `Reportes/mediciones_<etiqueta>.csv` |

La consulta de prueba: **total de `salary_in_usd` por nivel de experiencia**.

In [3]:
def medir(et):
    prueba_parquet = tools.RUTA_MODELADO_PRUEBAS / f'prueba_{et}.parquet'       # data lake
    prueba_duckdb  = tools.RUTA_MODELADO_PRUEBAS / f'prueba_{et}_duckdb.db'     # data warehouse
    mediciones_csv = tools.RUTA_MODELADO_REPORTES / f'mediciones_{et}.csv'

    # ── PASO 1: datos de prueba ───────────────────────────────────────────
    df_prueba = datos_prueba(et)
    filas_prueba = len(df_prueba)
    print(f'[{et}] filas originales x {MULTIPLICAR} = {filas_prueba:,} filas de prueba')

    # ── PASO 2: carga ─────────────────────────────────────────────────────
    _, seg_carga_parquet = A.cronometrar(lambda: df_prueba.to_parquet(prueba_parquet, index=False))

    con_duck = duckdb.connect(str(prueba_duckdb))
    con_duck.register('df_prueba', df_prueba)                     # el DataFrame visible para el SQL de DuckDB
    _, seg_carga_duckdb = A.cronometrar(
        lambda: con_duck.execute('CREATE OR REPLACE TABLE salarios AS SELECT * FROM df_prueba'))

    # ── PASO 3: disco ─────────────────────────────────────────────────────
    con_duck.execute('CHECKPOINT')                                # termina de escribir en el archivo
    mb_parquet, mb_duckdb = A.mb(prueba_parquet), A.mb(prueba_duckdb)

    # ── PASO 4: consulta (misma consulta en los dos) ──────────────────────
    consulta = """
        SELECT experience_level, SUM(salary_in_usd)::BIGINT AS salarios_usd
        FROM {origen}
        GROUP BY experience_level
        ORDER BY salarios_usd DESC
    """
    res_parquet, seg_consulta_parquet = A.cronometrar(
        lambda: con_duck.sql(consulta.format(origen=f"read_parquet('{prueba_parquet}')")).df())   # lakehouse
    res_duckdb, seg_consulta_duckdb = A.cronometrar(
        lambda: con_duck.sql(consulta.format(origen='salarios')).df())                           # warehouse
    con_duck.close()
    totales_iguales = int(res_parquet['salarios_usd'].sum()) == int(res_duckdb['salarios_usd'].sum())

    # ── PASO 5: consulta con JOIN sobre el esquema estrella (datos reales) ──
    con_serving = duckdb.connect(str(tools.SERVING_DB), read_only=True)   # solo lectura: no modifica el modelo
    res_estrella, seg_consulta_estrella = A.cronometrar(lambda: con_serving.sql(f"""
        SELECT x.codigo AS experience_level, x.nivel, SUM(f.salario_usd)::BIGINT AS salarios_usd
        FROM fact_salarios_{et}   AS f
        JOIN dim_experiencia_{et} AS x USING (experiencia_id)
        GROUP BY x.codigo, x.nivel
        ORDER BY salarios_usd DESC
    """).df())
    filas_estrella = con_serving.sql(f'SELECT COUNT(*) FROM fact_salarios_{et}').fetchall()[0][0]
    con_serving.close()

    # ── PASO 6: tabla de mediciones ───────────────────────────────────────
    mediciones = pd.DataFrame([
        {'destino': 'Parquet', 'estrategia': 'Data lake / lakehouse', 'filas': filas_prueba,
         'carga_seg': round(seg_carga_parquet, 2), 'disco_mb': round(mb_parquet, 1), 'consulta_seg': round(seg_consulta_parquet, 3)},
        {'destino': 'DuckDB', 'estrategia': 'Data warehouse', 'filas': filas_prueba,
         'carga_seg': round(seg_carga_duckdb, 2), 'disco_mb': round(mb_duckdb, 1), 'consulta_seg': round(seg_consulta_duckdb, 3)},
        {'destino': 'serving.db', 'estrategia': 'Esquema estrella (datos reales)', 'filas': filas_estrella,
         'carga_seg': None, 'disco_mb': None, 'consulta_seg': round(seg_consulta_estrella, 3)},
    ])
    mediciones.insert(0, 'dataset', et)
    mediciones['multiplicar'] = [MULTIPLICAR, MULTIPLICAR, 1]
    mediciones.to_csv(mediciones_csv, index=False)

    del df_prueba; gc.collect()
    return dict(mediciones=mediciones, res_parquet=res_parquet, res_duckdb=res_duckdb,
                res_estrella=res_estrella, totales_iguales=totales_iguales, csv=mediciones_csv)

---
## Dataset `adilshamim8`

In [4]:
r = medir('adilshamim8')
print(f"¿Parquet y DuckDB dan el mismo total? {r['totales_iguales']}  "
      f"(Parquet: {r['res_parquet']['salarios_usd'].sum():,} | DuckDB: {r['res_duckdb']['salarios_usd'].sum():,})")
print(f"Mediciones guardadas en: {r['csv'].relative_to(tools.BASE_DIR)}")
resultados_adils = r
r['mediciones']

[adilshamim8] filas originales x 20 = 1,438,260 filas de prueba


¿Parquet y DuckDB dan el mismo total? True  (Parquet: 217,409,183,200 | DuckDB: 217,409,183,200)
Mediciones guardadas en: Modelado_de_Datos/Reportes/mediciones_adilshamim8.csv


,dataset,destino,estrategia,filas,carga_seg,disco_mb,consulta_seg,multiplicar
0,adilshamim8,Parquet,Data lake / lakehouse,1438260,0.54,4.8,0.016,20
1,adilshamim8,DuckDB,Data warehouse,1438260,0.92,21.3,0.002,20
2,adilshamim8,serving.db,Esquema estrella (datos reales),71913,NaN,NaN,0.003,1


In [5]:
print('Consulta en Parquet (lakehouse)'); display(r['res_parquet'])
print('Consulta en DuckDB (warehouse)');  display(r['res_duckdb'])
print('Consulta en el esquema estrella (datos reales)'); display(r['res_estrella'])

Consulta en Parquet (lakehouse)


,experience_level,salarios_usd
0,SE,126533912680
1,MI,63759333960
2,EN,15633724300
3,EX,11482212260


Consulta en DuckDB (warehouse)


,experience_level,salarios_usd
0,SE,126533912680
1,MI,63759333960
2,EN,15633724300
3,EX,11482212260


Consulta en el esquema estrella (datos reales)


,experience_level,nivel,salarios_usd
0,SE,Senior,6326695634
1,MI,Intermedio (Mid),3187966698
2,EN,Inicial (Entry/Junior),781686215
3,EX,Ejecutivo / Director,574110613


---
## Dataset `samithsachidanandan`

In [6]:
r = medir('samithsachidanandan')
print(f"¿Parquet y DuckDB dan el mismo total? {r['totales_iguales']}  "
      f"(Parquet: {r['res_parquet']['salarios_usd'].sum():,} | DuckDB: {r['res_duckdb']['salarios_usd'].sum():,})")
print(f"Mediciones guardadas en: {r['csv'].relative_to(tools.BASE_DIR)}")
resultados_samit = r
r['mediciones']

[samithsachidanandan] filas originales x 20 = 884,880 filas de prueba


¿Parquet y DuckDB dan el mismo total? True  (Parquet: 133,894,233,060 | DuckDB: 133,894,233,060)
Mediciones guardadas en: Modelado_de_Datos/Reportes/mediciones_samithsachidanandan.csv


,dataset,destino,estrategia,filas,carga_seg,disco_mb,consulta_seg,multiplicar
0,samithsachidanandan,Parquet,Data lake / lakehouse,884880,0.30,2.5,0.006,20
1,samithsachidanandan,DuckDB,Data warehouse,884880,0.56,13.5,0.001,20
2,samithsachidanandan,serving.db,Esquema estrella (datos reales),44244,NaN,NaN,0.002,1


In [7]:
print('Consulta en Parquet (lakehouse)'); display(r['res_parquet'])
print('Consulta en DuckDB (warehouse)');  display(r['res_duckdb'])
print('Consulta en el esquema estrella (datos reales)'); display(r['res_estrella'])

Consulta en Parquet (lakehouse)


,experience_level,salarios_usd
0,SE,79095544940
1,MI,39421212320
2,EN,9918637240
3,EX,5458838560


Consulta en DuckDB (warehouse)


,experience_level,salarios_usd
0,SE,79095544940
1,MI,39421212320
2,EN,9918637240
3,EX,5458838560


Consulta en el esquema estrella (datos reales)


,experience_level,nivel,salarios_usd
0,SE,Senior,3954777247
1,MI,Intermedio (Mid),1971060616
2,EN,Inicial (Entry/Junior),495931862
3,EX,Ejecutivo / Director,272941928


---
## Dataset `saurabhbadole`

In [8]:
r = medir('saurabhbadole')
print(f"¿Parquet y DuckDB dan el mismo total? {r['totales_iguales']}  "
      f"(Parquet: {r['res_parquet']['salarios_usd'].sum():,} | DuckDB: {r['res_duckdb']['salarios_usd'].sum():,})")
print(f"Mediciones guardadas en: {r['csv'].relative_to(tools.BASE_DIR)}")
resultados_saura = r
r['mediciones']

[saurabhbadole] filas originales x 20 = 932,740 filas de prueba


¿Parquet y DuckDB dan el mismo total? True  (Parquet: 141,227,860,980 | DuckDB: 141,227,860,980)
Mediciones guardadas en: Modelado_de_Datos/Reportes/mediciones_saurabhbadole.csv


,dataset,destino,estrategia,filas,carga_seg,disco_mb,consulta_seg,multiplicar
0,saurabhbadole,Parquet,Data lake / lakehouse,932740,0.32,2.9,0.007,20
1,saurabhbadole,DuckDB,Data warehouse,932740,0.59,14.0,0.001,20
2,saurabhbadole,serving.db,Esquema estrella (datos reales),46637,NaN,NaN,0.002,1


In [9]:
print('Consulta en Parquet (lakehouse)'); display(r['res_parquet'])
print('Consulta en DuckDB (warehouse)');  display(r['res_duckdb'])
print('Consulta en el esquema estrella (datos reales)'); display(r['res_estrella'])

Consulta en Parquet (lakehouse)


,experience_level,salarios_usd
0,SE,83411362080
1,MI,41480908120
2,EN,10533269340
3,EX,5802321440


Consulta en DuckDB (warehouse)


,experience_level,salarios_usd
0,SE,83411362080
1,MI,41480908120
2,EN,10533269340
3,EX,5802321440


Consulta en el esquema estrella (datos reales)


,experience_level,nivel,salarios_usd
0,SE,Senior,4170568104
1,MI,Intermedio (Mid),2074045406
2,EN,Inicial (Entry/Junior),526663467
3,EX,Ejecutivo / Director,290116072


---
## Resumen de los tres datasets

In [10]:
resumen = pd.concat([pd.read_csv(tools.RUTA_MODELADO_REPORTES / f'mediciones_{et}.csv') for et in tools.ETIQUETAS],
                    ignore_index=True)
resumen

,dataset,destino,estrategia,filas,carga_seg,disco_mb,consulta_seg,multiplicar
0,adilshamim8,Parquet,Data lake / lakehouse,1438260,0.54,4.8,0.016,20
1,adilshamim8,DuckDB,Data warehouse,1438260,0.92,21.3,0.002,20
2,adilshamim8,serving.db,Esquema estrella (datos reales),71913,NaN,NaN,0.003,1
3,samithsachidanandan,Parquet,Data lake / lakehouse,884880,0.30,2.5,0.006,20
4,samithsachidanandan,DuckDB,Data warehouse,884880,0.56,13.5,0.001,20
5,samithsachidanandan,serving.db,Esquema estrella (datos reales),44244,NaN,NaN,0.002,1
6,saurabhbadole,Parquet,Data lake / lakehouse,932740,0.32,2.9,0.007,20
7,saurabhbadole,DuckDB,Data warehouse,932740,0.59,14.0,0.001,20
8,saurabhbadole,serving.db,Esquema estrella (datos reales),46637,NaN,NaN,0.002,1


In [11]:
# ── ¿Quién ganó en cada medición? ──────────────────────────────────────────
pruebas = resumen[resumen['destino'] != 'serving.db']
ganadores = []
for et, g in pruebas.groupby('dataset', sort=False):
    g = g.set_index('destino')
    ganadores.append({'dataset': et,
                      'carga_mas_rapida': g['carga_seg'].idxmin(),
                      'menos_disco': g['disco_mb'].idxmin(),
                      'consulta_mas_rapida': g['consulta_seg'].idxmin(),
                      'parquet_vs_duckdb_disco': f"{g.loc['Parquet', 'disco_mb'] / g.loc['DuckDB', 'disco_mb']:.0%}"})
pd.DataFrame(ganadores)

,dataset,carga_mas_rapida,menos_disco,consulta_mas_rapida,parquet_vs_duckdb_disco
0,adilshamim8,Parquet,Parquet,DuckDB,23%
1,samithsachidanandan,Parquet,Parquet,DuckDB,19%
2,saurabhbadole,Parquet,Parquet,DuckDB,21%


### Cómo leer los resultados

| Medición | Suele ganar | Por qué |
|---|---|---|
| Disco | Parquet | Solo guarda los datos, por columnas y comprimidos |
| Carga | DuckDB | No tiene que comprimir un archivo; copia el DataFrame a su formato interno |
| Consulta | DuckDB, por poco | Los datos ya están dentro de la base, con sus estadísticas |

La carpeta `Pruebas/` se puede borrar al terminar: solo contiene copias multiplicadas para medir. Se conservan los `Reportes/mediciones_<etiqueta>.csv`.

### Nuestros resultados

En los tres datasets **Parquet cargó más rápido** (0.54 s contra 0.92 s en `adilshamim8`) y **ocupó entre 19 % y 23 % del espacio** de DuckDB (4.8 MB contra 21.3 MB): guarda por columnas, con diccionario y compresión, y los valores repetidos casi no ocupan espacio. **DuckDB respondió más rápido** (0.002 s contra 0.016 s) porque los datos ya están dentro de la base. La consulta con JOIN sobre el esquema estrella tardó entre 0.002 y 0.003 s con los datos reales. Son tiempos de una sola corrida: varían unas milésimas en cada ejecución.